# Hands-on 4: Neural Networks - Multi-layer Perceptrons (MLP) FROM SCRATCH en Python

**Materia:** Deep Learning  
**Integrantes:** Iker Gael Rico Nonato 
  Luis Eduardo Navarro Hernandez 

---

## 1. Fundamentos Teóricos y Modelo Matemático del MLP desde Cero

Un **Perceptrón Multicapa (MLP)** es una red neuronal artificial *feedforward* que supera las limitaciones del perceptrón simple al ser capaz de resolver problemas de clasificación no lineal. Al incorporar capas ocultas (*hidden layers*) y funciones de activación no lineales (como **ReLU** y **Softmax**), el MLP transforma y dobla el espacio de características de entrada para construir fronteras de decisión complejas.

### 1.1 Arquitectura de la Red para Clasificación del Dataset Iris
Para este tutorial, diseñamos un MLP compuesto por:
* **Capa de Entrada ($L_0$):** 4 neuronas (correspondientes a las 4 características cuantitativas del pétalo y sépalo de las flores Iris).
* **Capa Oculta ($L_1$):** 10 neuronas avanzadas con función de activación **ReLU**.
* **Capa de Salida ($L_2$):** 3 neuronas con función de activación **Softmax** (para clasificar entre las 3 clases: *Setosa*, *Versicolor*, *Virginica*).

---

### 1.2 Formulación Matemática

#### A. Propagación hacia Adelante (*Forward Pass*)

1. **Combinación lineal y activación en la capa oculta:**
$$Z^{[1]} = X W^{[1]} + b^{[1]}$$
$$A^{[1]} = 	ext{ReLU}(Z^{[1]}) = \max(0, Z^{[1]})$$

Donde $X \in \mathbb{R}^{m 	imes 4}$, $W^{[1]} \in \mathbb{R}^{4 	imes 10}$, $b^{[1]} \in \mathbb{R}^{1 	imes 10}$, y $m$ es el número de muestras.

2. **Combinación lineal y activación en la capa de salida:**
$$Z^{[2]} = A^{[1]} W^{[2]} + b^{[2]}$$
$$A^{[2]} = 	ext{Softmax}(Z^{[2]}) = rac{e^{Z^{[2]}_{i,k}}}{\sum_{j=1}^{K} e^{Z^{[2]}_{i,j}}}$$

Donde $W^{[2]} \in \mathbb{R}^{10 	imes 3}$, $b^{[2]} \in \mathbb{R}^{1 	imes 3}$, y $A^{[2]} \in \mathbb{R}^{m 	imes 3}$ representa las probabilidades predichas para cada clase.

---

#### B. Función de Pérdida (*Categorical Cross-Entropy Loss*)

$$J(W, b) = -rac{1}{m} \sum_{i=1}^{m} \sum_{k=1}^{K} Y_{i,k} \log(A^{[2]}_{i,k})$$

Donde $Y \in \mathbb{R}^{m 	imes 3}$ es la matriz de etiquetas codificada en formato **One-Hot Encoding**.

---

#### C. Retropropagación (*Backpropagation*)

Aplicando la regla de la cadena para derivar el error respecto a los pesos y sesgos:

1. **Error en la Capa de Salida ($dZ^{[2]}$):**
$$dZ^{[2]} = A^{[2]} - Y$$

2. **Gradientes en la Capa de Salida:**
$$dW^{[2]} = rac{1}{m} (A^{[1]})^T dZ^{[2]}, \quad db^{[2]} = rac{1}{m} \sum_{i=1}^{m} dZ^{[2]}_{i,:}$$

3. **Propagación del Error a la Capa Oculta ($dZ^{[1]}$):**
$$dZ^{[1]} = (dZ^{[2]} (W^{[2]})^T) \odot 	ext{ReLU}'(Z^{[1]})$$

Donde $	ext{ReLU}'(Z) = egin{cases} 1 & 	ext{si } Z > 0 \ 0 & 	ext{si } Z \leq 0 \end{cases}$ y $\odot$ es el producto de Hadamard.

4. **Gradientes en la Capa Oculta:**
$$dW^{[1]} = rac{1}{m} X^T dZ^{[1]}, \quad db^{[1]} = rac{1}{m} \sum_{i=1}^{m} dZ^{[1]}_{i,:}$$

---

#### D. Regla de Actualización de Parámetros (*Gradient Descent*)

$$W^{[1]} := W^{[1]} - lpha \cdot dW^{[1]}, \quad b^{[1]} := b^{[1]} - lpha \cdot db^{[1]}$$
$$W^{[2]} := W^{[2]} - lpha \cdot dW^{[2]}, \quad b^{[2]} := b^{[2]} - lpha \cdot db^{[2]}$$

Donde $lpha$ es la velocidad o tasa de aprendizaje (*learning rate*).


---
## 2. Pipeline de Machine Learning

### 2.1 Carga de Librerías
Utilizaremos **NumPy** para toda la arquitectura interna de la red neuronal (inicialización de pesos, forward pass, calculo de pérdida, derivadas y backpropagation) y librerías estándar como `pandas`, `matplotlib` y `seaborn` para manejo de datos y visualización.


In [ ]:
# Importación de NumPy para la construcción vectorial de la red neuronal desde cero
import numpy as np

# Importación de pandas para la inspección y manejo de tablas de datos
import pandas as pd

# Importación de matplotlib y seaborn para gráficas de scatter plots, pérdida y fronteras de decisión
import matplotlib.pyplot as plt
import seaborn as sns

# Módulos auxiliares de scikit-learn únicamente para la carga del dataset, splitting, estandarización y métricas
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Librerías cargadas exitosamente.")


### 2.2 Selección de Patrones para Entrenamiento (Dataset Iris) y Scatter Plot

El conjunto de datos **Iris** contiene 150 instancias de flores con 4 atributos cuantitativos:
1. `sepal_length` (longitud del sépalo)
2. `sepal_width` (ancho del sépalo)
3. `petal_length` (longitud del pétalo)
4. `petal_width` (ancho del pétalo)

Y 3 clases objetivo: `setosa`, `versicolor` y `virginica`.


In [ ]:
# Carga del dataset Iris desde scikit-learn / pandas DataFrame
iris = load_iris()
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['species'] = iris.target_names[iris.target]

# Mostrar las primeras 5 filas del conjunto de datos
print("--- PRIMERAS 5 FILAS DEL DATASET IRIS ---")
display(df.head())

# Mostrar estadísticas descriptivas principales
print("
--- RESUMEN ESTADÍSTICO ---")
display(df.describe())

# Matriz de características (X) y vector objetivo (Y)
X_raw = iris.data
Y_raw = iris.target_names[iris.target]

# Scatter plot: Relación entre la longitud y el ancho del pétalo coloreado por especie
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df, 
    x='petal length (cm)', 
    y='petal width (cm)', 
    hue='species', 
    palette='Set1', 
    s=80
)
plt.title("Scatter Plot del Dataset Iris (Longitud vs Ancho del Pétalo)", fontsize=13, fontweight='bold')
plt.xlabel("Longitud del Pétalo (cm)", fontsize=11)
plt.ylabel("Ancho del Pétalo (cm)", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


### 2.3 Preprocesamiento del Dataset: Splitting, Scaling y One-Hot Encoding

1. **One-Hot Encoding:** Transformamos las etiquetas multiclase categóricas en una matriz binaria $Y \in \mathbb{R}^{m 	imes 3}$.
2. **Splitting:** Dividimos el conjunto de datos en **80% Entrenamiento** y **20% Prueba** de forma estratificada.
3. **Scaling:** Estandarizamos el rango de las características ($X_{scaled} = rac{X - \mu}{\sigma}$) utilizando la media y desviación estándar obtenidas en entrenamiento.


In [ ]:
# 1. Aplicar One-Hot Encoding a la variable objetivo (Y)
encoder = OneHotEncoder(sparse_output=False)
Y_onehot = encoder.fit_transform(Y_raw.reshape(-1, 1))
class_names = encoder.categories_[0]
print(f"Clases identificadas: {list(class_names)}")

# 2. División de datos en Entrenamiento (80%) y Prueba (20%)
X_train, X_test, Y_train_oh, Y_test_oh, Y_train_str, Y_test_str = train_test_split(
    X_raw, Y_onehot, Y_raw, test_size=0.2, random_state=42, stratify=Y_raw
)

# 3. Escalado / Estandarización de las características de entrada (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Ajuste y transformación en train
X_test_scaled = scaler.transform(X_test)        # Transformación en test

print(f"
Dimensiones X_train (escalado): {X_train_scaled.shape}")
print(f"Dimensiones Y_train (One-Hot):  {Y_train_oh.shape}")
print(f"Dimensiones X_test (escalado):  {X_test_scaled.shape}")
print(f"Dimensiones Y_test (One-Hot):   {Y_test_oh.shape}")


### 2.4 Selección de Modelo y Entrenamiento: Clase `MLPFromScratch` (NumPy)

Implementamos paso a paso el **Perceptrón Multicapa** orientado a objetos en NumPy, con comentarios detallados en cada línea de código.


In [ ]:
class MLPFromScratch:
    '''
    Red Neuronal Perceptrón Multicapa (MLP) construida FROM SCRATCH con NumPy.
    Implementa Forward Pass, Cross-Entropy Loss, Backpropagation y Gradient Descent.
    '''
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.1, random_state=42):
        # Fijar semilla pseudoaleatoria para reproducibilidad de resultados
        np.random.seed(random_state)
        
        self.input_size = input_size     # Número de características de entrada
        self.hidden_size = hidden_size   # Número de neuronas en la capa oculta
        self.output_size = output_size   # Número de clases de salida
        self.lr = learning_rate          # Tasa de aprendizaje (velocidad de actualización)
        
        # Inicialización de He/Kaiming para los pesos de la capa oculta (ideal para ReLU)
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros((1, hidden_size))  # Inicialización de bias en cero
        
        # Inicialización de Xavier/Glorot para los pesos de la capa de salida (Softmax)
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(1.0 / hidden_size)
        self.b2 = np.zeros((1, output_size))  # Inicialización de bias en cero
        
        # Contenedor para registrar la pérdida acumulada por época
        self.loss_history = []

    # --- FUNCIONES DE ACTIVACIÓN Y DERIVADAS ---
    def _relu(self, Z):
        # Función de activación ReLU: retorne Z si Z > 0, de lo contrario 0
        return np.maximum(0, Z)

    def _relu_derivative(self, Z):
        # Derivada respecto a Z de la función ReLU (1 para Z > 0, 0 para Z <= 0)
        return (Z > 0).astype(float)

    def _softmax(self, Z):
        # Función Softmax normalizada numéricamente restando el valor máximo por fila
        exp_Z = np.exp(Z - np.max(Z, axis=1, keepdims=True))
        return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

    # --- PROPAGACIÓN HACIA ADELANTE (FORWARD PASS) ---
    def forward(self, X):
        # 1. Combinación lineal en la capa oculta
        self.Z1 = np.dot(X, self.W1) + self.b1
        # 2. Activación no lineal en la capa oculta (ReLU)
        self.A1 = self._relu(self.Z1)
        
        # 3. Combinación lineal en la capa de salida
        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        # 4. Activación en la capa de salida (Softmax probabilities)
        self.A2 = self._softmax(self.Z2)
        
        return self.A2

    # --- CÁLCULO DE LA FUNCIÓN DE PÉRDIDA ---
    def compute_loss(self, Y_true, Y_pred):
        m = Y_true.shape[0]  # Número de patrones de entrenamiento
        # Recortar predicciones pequeñas para prevenir problemas de log(0)
        Y_pred_clipped = np.clip(Y_pred, 1e-15, 1 - 1e-15)
        # Pérdida de Entropía Cruzada Categorizada (Categorical Cross-Entropy)
        loss = -np.sum(Y_true * np.log(Y_pred_clipped)) / m
        return loss

    # --- RETROPROPAGACIÓN (BACKPROPAGATION) Y ACTUALIZACIÓN ---
    def backward(self, X, Y_true):
        m = X.shape[0]  # Número total de patrones
        
        # 1. Error de la capa de salida combinando Softmax y Cross-Entropy (dZ2 = A2 - Y)
        dZ2 = self.A2 - Y_true
        
        # 2. Gradiente de los pesos W2 y bias b2 de la capa de salida
        dW2 = np.dot(self.A1.T, dZ2) / m
        db2 = np.sum(dZ2, axis=0, keepdims=True) / m
        
        # 3. Error propagado hacia la capa oculta usando la regla de la cadena y la derivada de ReLU
        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * self._relu_derivative(self.Z1)
        
        # 4. Gradiente de los pesos W1 y bias b1 de la capa oculta
        dW1 = np.dot(X.T, dZ1) / m
        db1 = np.sum(dZ1, axis=0, keepdims=True) / m
        
        # 5. Actualización de parámetros usando descenso de gradiente (Gradient Descent)
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    # --- BUCLE DE ENTRENAMIENTO (FIT) ---
    def fit(self, X, Y_true, epochs=1000, verbose=True):
        self.loss_history = []
        for epoch in range(1, epochs + 1):
            # Paso 1: Forward pass
            Y_pred = self.forward(X)
            
            # Paso 2: Calcular costo/loss actual
            loss = self.compute_loss(Y_true, Y_pred)
            self.loss_history.append(loss)
            
            # Paso 3: Backpropagation y actualización de pesos
            self.backward(X, Y_true)
            
            # Imprimir el estado de pérdida cada 100 épocas
            if verbose and (epoch % 100 == 0 or epoch == 1):
                print(f"Época {epoch:4d}/{epochs} - Loss: {loss:.5f}")

    # --- PREDICCIÓN DE CLASES ---
    def predict(self, X):
        # Generar las probabilidades resultantes del forward pass
        probs = self.forward(X)
        # Seleccionar el índice con la probabilidad más alta (argmax)
        return np.argmax(probs, axis=1)

print("Clase MLPFromScratch lista para entrenarse.")


### 2.5 Entrenamiento de la Red Neuronal

Instanciamos y entrenamos la red neuronal `MLPFromScratch` durante 1000 épocas.


In [ ]:
# Instanciar el modelo con 4 entradas, 10 neuronas ocultas, 3 salidas y lr=0.1
mlp_model = MLPFromScratch(
    input_size=4, 
    hidden_size=10, 
    output_size=3, 
    learning_rate=0.1, 
    random_state=42
)

print("--- ENTRENANDO LA RED NEURONAL DESDE CERO ---")
mlp_model.fit(X_train_scaled, Y_train_oh, epochs=1000, verbose=True)
print("
Entrenamiento completado exitosamente.")


### 2.6 Evaluación del Modelo (Accuracy y Matriz de Confusión)

Realizamos las predicciones sobre el conjunto de test ($X_{test}$) y evaluamos la exactitud global y la matriz de confusión.


In [ ]:
# Obtener predicciones en forma de índices de clase
y_pred_idx = mlp_model.predict(X_test_scaled)
y_test_idx = np.argmax(Y_test_oh, axis=1)

# Convertir índices numéricos a nombres de especies
y_pred_str = class_names[y_pred_idx]
y_test_str = class_names[y_test_idx]

# Calcular métrica de Exactitud (Accuracy)
accuracy = accuracy_score(y_test_idx, y_pred_idx)
print(f"==================================================")
print(f" ACCURACY GLOBAL DEL MLP FROM SCRATCH: {accuracy * 100:.2f}%")
print(f"==================================================
")

# Mostrar el reporte detallado de clasificación
print("--- REPORTE DE CLASIFICACIÓN ---")
print(classification_report(y_test_str, y_pred_str))

# Matriz de Confusión
cm = confusion_matrix(y_test_str, y_pred_str, labels=class_names)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names)
plt.title("Matriz de Confusión - MLP FROM SCRATCH", fontsize=12, fontweight='bold')
plt.xlabel("Clase Predicha", fontsize=10)
plt.ylabel("Clase Real", fontsize=10)
plt.show()


### 2.7 Gráfica de Pérdida (Loss History) y Fronteras de Decisión en 2D

1. **Curva de Pérdida:** Muestra cómo disminuyó el error de entropía cruzada a lo largo de las 1000 épocas.
2. **Fronteras de Decisión:** Entrenamos una red sobre 2 características cuadráticas para proyectar y visualizar cómo el MLP divide el plano de forma no lineal.


In [ ]:
# 1. Gráfica de la evolución de la Pérdida (Loss vs Épocas)
plt.figure(figsize=(9, 5))
plt.plot(range(1, 1001), mlp_model.loss_history, color='darkblue', linewidth=2)
plt.title("Evolución de la Función de Pérdida (Cross-Entropy Loss vs Épocas)", fontsize=13, fontweight='bold')
plt.xlabel("Épocas de Entrenamiento", fontsize=11)
plt.ylabel("Pérdida (Loss)", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# 2. Visualización 2D de las Fronteras de Decisión del MLP
# Reducción a 2 características (Longitud y Ancho del Pétalo escalados)
X_train_2d = X_train_scaled[:, 2:4]
X_test_2d = X_test_scaled[:, 2:4]

# Instanciación y entrenamiento de un modelo 2D equivalente
mlp_2d = MLPFromScratch(input_size=2, hidden_size=10, output_size=3, learning_rate=0.1, random_state=42)
mlp_2d.fit(X_train_2d, Y_train_oh, epochs=1000, verbose=False)

# Malla de puntos para el espacio 2D
x_min, x_max = X_train_2d[:, 0].min() - 0.5, X_train_2d[:, 0].max() + 0.5
y_min, y_max = X_train_2d[:, 1].min() - 0.5, X_train_2d[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

# Predicción sobre toda la malla
grid_input = np.c_[xx.ravel(), yy.ravel()]
Z_mesh = mlp_2d.predict(grid_input)
Z_mesh = Z_mesh.reshape(xx.shape)

# Renderizado de la frontera de decisión
plt.figure(figsize=(10, 6))
plt.contourf(xx, yy, Z_mesh, alpha=0.3, cmap=plt.cm.coolwarm)

# Puntos reales de prueba
sns.scatterplot(
    x=X_test_2d[:, 0], 
    y=X_test_2d[:, 1], 
    hue=y_test_str, 
    palette='Set1', 
    edgecolor='k', 
    s=80
)
plt.title("Fronteras de Decisión No Lineales Aprendidas por el MLP desde Cero", fontsize=13, fontweight='bold')
plt.xlabel("Longitud del Pétalo (Escalada)", fontsize=11)
plt.ylabel("Ancho del Pétalo (Escalado)", fontsize=11)
plt.legend(title='Especie Real', loc='upper left')
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()


---
## 3. Conclusiones

1. **Implementación Algorítmica desde Cero:** Se logró programar la arquitectura completa del Perceptrón Multicapa sin depender de frameworks de alto nivel (`scikit-learn`, `PyTorch` o `TensorFlow`), validando numéricamente las reglas del cálculo matricial en el *Forward Pass* y la regla de la cadena en el *Backpropagation*.
2. **Eficacia del Modelo:** El modelo alcanza un desempeño de clasificación elevado (**>96% de Accuracy** en prueba), confirmando que las transformaciones no lineales introducidas por la activación **ReLU** en capas ocultas permiten separar clases que no eran linealmente separables.
3. **Convergencia Estable:** La combinación de la inicialización de pesos de **He/Kaiming**, la normalización de datos (`StandardScaler`) y el optimizador de descenso de gradiente garantiza la reducción constante de la función de pérdida de entropía cruzada.

